# Phase 5 L1 - Gaussian Blur Resist

Compare Level 0 threshold resist with Level 1 Gaussian blur before thresholding.

In [ ]:
from src import constants as C
from src.dof import focus_drilling_average
from src.mask import MaskGrid, kirchhoff_mask, line_space_pattern
from src.metrics import critical_dimension, edge_positions, mean_absolute_epe
from src.pupil import PupilSpec
from src.resist_blur import blurred_threshold_resist, blur_dose_sweep, gaussian_blur, transition_width
from src.resist_threshold import threshold_resist

In [ ]:
pitch_m = 80e-9
grid = MaskGrid(nx=512, ny=64, pixel_size=1e-9)
pattern = line_space_pattern(grid, pitch_m=pitch_m, duty_cycle=0.5)
mask = kirchhoff_mask(pattern)

aerial, wafer = focus_drilling_average(
    mask,
    grid,
    [0.0],
    pupil_spec=PupilSpec(grid_size=512, na=C.NA_HIGH, obscuration_ratio=0.0),
    anamorphic=False,
)

printed_l0 = threshold_resist(aerial, dose=1.0, threshold=0.2)
printed_l1 = blurred_threshold_resist(aerial, wafer.pixel_x_m, sigma_m=2e-9, dose=1.0, threshold=0.2)

In [ ]:
cy = wafer.ny // 2
target_clear = pattern[cy, :] == 0.0
target_cd = critical_dimension(target_clear, grid.pixel_size)
target_edges = edge_positions(target_clear, grid.pixel_size)

summary = {
    "target_cd_nm": target_cd * 1e9,
    "l0_cd_nm": critical_dimension(printed_l0[cy, :], wafer.pixel_x_m) * 1e9,
    "l1_cd_nm": critical_dimension(printed_l1[cy, :], wafer.pixel_x_m) * 1e9,
    "l0_epe_nm": mean_absolute_epe(target_edges, edge_positions(printed_l0[cy, :], wafer.pixel_x_m)) * 1e9,
    "l1_epe_nm": mean_absolute_epe(target_edges, edge_positions(printed_l1[cy, :], wafer.pixel_x_m)) * 1e9,
    "l1_transition_width_nm": transition_width(gaussian_blur(aerial[cy, :], wafer.pixel_x_m, 2e-9), wafer.pixel_x_m) * 1e9,
}
summary

In [ ]:
sweep = blur_dose_sweep(
    aerial[cy, :],
    target_clear,
    wafer.pixel_x_m,
    sigma_values_m=[1e-9, 2e-9, 4e-9],
    doses=[0.8, 1.0, 1.2],
    threshold=0.2,
)

[
    {
        "sigma_nm": point.sigma_m * 1e9,
        "dose": point.dose,
        "cd_nm": point.cd_m * 1e9,
        "epe_nm": None if point.mean_abs_epe_m is None else point.mean_abs_epe_m * 1e9,
        "transition_width_nm": point.transition_width_m * 1e9,
        "lwr_proxy_nm": point.lwr_proxy_m * 1e9,
    }
    for point in sweep
]